In [1]:
from typing import TypeAlias, Union

import torch
from torch import nn
import kornia.augmentation as K

from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets, transforms

TensorPair: TypeAlias = tuple[torch.Tensor, torch.Tensor]
DeviceLikeType: TypeAlias = Union[str, torch.device, int]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
NORMALIZE_MEAN = 0.1307
NORMALIZE_STD = 0.3081

BATCH_SIZE = 128

In [3]:
train_transform = transforms.Compose(
    [
        transforms.RandomAffine(
            degrees=10,
            translate=(0.08, 0.08),
            scale=(0.90, 1.10),
        ),
        transforms.ToTensor(),
        transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
        transforms.Lambda(torch.flatten),
    ]
)

train_dataset = datasets.MNIST(
    root="./data", train=True, download=True, transform=train_transform
)

pin_memory = device.type == "cuda"

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=pin_memory,
)

In [4]:
it = iter(train_loader)
%timeit -n 400 -r 1 next(it)

23.7 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 400 loops each)


In [3]:
class MNISTCollator:
    def __init__(self, device: DeviceLikeType):
        self.device = torch.device(device)

        self.transform = nn.Sequential(
            K.RandomAffine(
                degrees=10,
                translate=(0.08, 0.08),
                scale=(0.90, 1.10),
                p=1.0,
            ),
            K.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
            nn.Flatten(start_dim=1),
        ).to(self.device)

    def __call__(self, batch: list[TensorPair]) -> TensorPair:
        images, targets = zip(*batch)

        images = torch.stack(images).unsqueeze(1)
        images = images.to(self.device, dtype=torch.float32) / 255.0

        targets = torch.stack(targets).to(self.device)

        images = self.transform(images)

        return images, targets


train_raw = datasets.MNIST(root="./data", train=True, download=True)

train_dataset = TensorDataset(train_raw.data, train_raw.targets.long())

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=MNISTCollator(device=device),
)

In [4]:
it = iter(train_loader)
%timeit -n 400 -r 1 next(it)

5.57 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 400 loops each)
